# Gradient Descent from Scratch: How Neural Networks Actually Learn

Every neural network training run, every fine-tune, every RLHF reward model — they all reduce to gradient descent. This notebook builds the algorithm from scratch using NumPy so you understand every part before PyTorch hides it.

There is no magic here. By the end you will have implemented:
- The gradient descent update rule
- 2D loss surface navigation
- Numerical gradient verification
- Mini-batch gradient descent for linear regression
- SGD with momentum
- Adam in 20 lines

**5 exercises.** All NumPy, no network calls, runs fully offline.

### What you'll learn
| Concept | Why it matters |
|---------|----------------|
| Gradient = direction of steepest ascent | Why we follow the negative gradient |
| Learning rate | Why tuning it is 80% of training |
| 2D loss surfaces | Where saddle points and ravines come from |
| Numerical gradients | How to debug autograd |
| Mini-batch GD | Why GPU training uses batches |
| Momentum + Adam | The real optimizers PyTorch ships |

### Prerequisites
- Notebook 01 (NumPy arrays, shapes, broadcasting)
- Basic calculus: partial derivatives, chain rule at a high level

```bash
pip install numpy matplotlib
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Color palette used throughout
BLUE   = '#4C9BE8'
GREEN  = '#5DBE7C'
ORANGE = '#E8A040'
RED    = '#E8704C'
PURPLE = '#9B59B6'

print('Setup complete. NumPy:', np.__version__)

---
## Part 1: What Is a Gradient?

A **gradient** is the direction and rate of steepest ascent in a loss surface. At any point in parameter space, the gradient tells you which way is "uphill."

**Gradient descent** follows the *negative* gradient — downhill — to minimize the loss.

This is it. Everything else is engineering detail.

Let's make it concrete with the simplest possible loss function:

$$L(w) = (w - 3)^2$$

This has an obvious minimum at $w = 3$. The gradient (derivative) is:

$$\frac{dL}{dw} = 2(w - 3)$$

When $w < 3$: gradient is negative → the function is sloping down toward 3 → we should move right (positive direction).
When $w > 3$: gradient is positive → the function is sloping up → we should move left.

Following the *negative* gradient always points us toward $w = 3$.

In [ ]:
def L(w):
    """Loss function: L(w) = (w - 3)^2"""
    return (w - 3) ** 2

def dL(w):
    """Analytical gradient: dL/dw = 2*(w - 3)"""
    return 2 * (w - 3)

# Demonstrate the gradient at a few points
test_points = [0.0, 1.5, 3.0, 5.0]
print('w       L(w)    dL/dw   direction')
print('-' * 45)
for w in test_points:
    grad = dL(w)
    direction = 'at minimum' if grad == 0 else ('move right (↑w)' if grad < 0 else 'move left (↓w)')
    print(f'{w:6.1f}  {L(w):6.2f}  {grad:+7.2f}  {direction}')

print()
print('Negative gradient always points toward the minimum at w=3.')

In [ ]:
w_range = np.linspace(-1, 7, 300)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: the parabola with gradient arrow at w=0.5
w_current = 0.5
grad = dL(w_current)
lr_demo = 0.3

ax = axes[0]
ax.plot(w_range, L(w_range), color=BLUE, lw=2.5, label='$L(w) = (w-3)^2$')
ax.scatter([w_current], [L(w_current)], color=ORANGE, s=100, zorder=5, label=f'Current point w={w_current}')

# Gradient arrow (direction of steepest ascent — uphill)
arrow_len = grad * 0.6
ax.annotate('', xy=(w_current + arrow_len, L(w_current)),
            xytext=(w_current, L(w_current)),
            arrowprops=dict(arrowstyle='->', color=RED, lw=2))
ax.text(w_current + arrow_len / 2, L(w_current) + 1.2, 'gradient\n(uphill)',
        ha='center', color=RED, fontsize=9, fontweight='bold')

# Negative gradient arrow (the step we take — downhill)
neg_arrow_len = -grad * lr_demo
w_new = w_current + neg_arrow_len
ax.annotate('', xy=(w_new, L(w_current)),
            xytext=(w_current, L(w_current)),
            arrowprops=dict(arrowstyle='->', color=GREEN, lw=2))
ax.text(w_new - 0.05, L(w_current) - 1.5, 'step taken\n(−lr × grad)',
        ha='center', color=GREEN, fontsize=9, fontweight='bold')

ax.axvline(3, color='gray', lw=1, ls='--', alpha=0.6, label='minimum at w=3')
ax.set_xlabel('w')
ax.set_ylabel('L(w)')
ax.set_title('The gradient points uphill.\nWe step in the opposite direction.', fontweight='bold')
ax.legend(fontsize=9)

# Right: gradient magnitude as a function of w
ax = axes[1]
ax.axhline(0, color='gray', lw=1)
ax.plot(w_range, dL(w_range), color=PURPLE, lw=2.5, label='$dL/dw = 2(w-3)$')
ax.fill_between(w_range, dL(w_range), 0,
                where=(dL(w_range) < 0), alpha=0.15, color=GREEN,
                label='negative → move right')
ax.fill_between(w_range, dL(w_range), 0,
                where=(dL(w_range) > 0), alpha=0.15, color=RED,
                label='positive → move left')
ax.scatter([3], [0], color=ORANGE, s=100, zorder=5, label='zero gradient = minimum')
ax.set_xlabel('w')
ax.set_ylabel('dL/dw')
ax.set_title('Gradient sign tells you which way to step', fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## Part 2: The Update Rule

The gradient descent update rule is:

$$w_{\text{new}} = w - \alpha \cdot \frac{dL}{dw}$$

where $\alpha$ (alpha) is the **learning rate** — a positive scalar that controls step size.

- **Too large**: you overshoot the minimum and can oscillate or diverge
- **Too small**: convergence is painfully slow
- **Just right**: smooth convergence in a reasonable number of steps

Let's implement this and watch all three behaviors.

In [ ]:
def gradient_descent_1d(w_init, lr, n_steps):
    """
    Run gradient descent on L(w) = (w - 3)^2 for n_steps.

    Returns:
        history: list of (w, loss) tuples, one per step
    """
    w = w_init
    history = []
    for _ in range(n_steps):
        loss = L(w)
        grad = dL(w)
        history.append((w, loss))
        w = w - lr * grad   # THE update rule
    return history

# Quick sanity check
history = gradient_descent_1d(w_init=0.0, lr=0.1, n_steps=5)
print('Step  w         loss')
print('-' * 30)
for i, (w, loss) in enumerate(history):
    print(f'{i:4d}  {w:8.4f}  {loss:.4f}')
print()
print(f'Starting at w=0.0, after 5 steps with lr=0.1 we reach w={history[-1][0]:.4f}')
print(f'Minimum is at w=3.0 — we are converging.')

In [ ]:
# Compare three learning rates: good, borderline, diverging
learning_rates = [0.1, 0.5, 1.1]
lr_colors      = [BLUE, GREEN, RED]
lr_labels      = ['lr=0.1 (converges)', 'lr=0.5 (fast but overshoots)', 'lr=1.1 (diverges)']

n_steps = 20
w_init  = 0.0

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Left: trajectory on the loss curve
ax = axes[0]
ax.plot(w_range, L(w_range), color='#888', lw=2, alpha=0.5, zorder=1)
ax.axvline(3, color='gray', lw=1, ls='--', alpha=0.5)

for lr, color, label in zip(learning_rates, lr_colors, lr_labels):
    hist = gradient_descent_1d(w_init, lr, n_steps)
    ws   = [h[0] for h in hist]
    ls   = [h[1] for h in hist]
    # Clip for display if diverging
    ws_clipped = np.clip(ws, -5, 11)
    ls_clipped = np.clip(ls, 0, 60)
    ax.scatter(ws_clipped[:8], ls_clipped[:8], color=color, s=30, zorder=5, alpha=0.8)
    ax.plot(ws_clipped[:8], ls_clipped[:8], color=color, lw=1.5, alpha=0.7, label=label)

ax.set_xlim(-1.5, 10)
ax.set_ylim(-1, 55)
ax.set_xlabel('w')
ax.set_ylabel('L(w)')
ax.set_title('Trajectory on the loss surface\n(first 8 steps)', fontweight='bold')
ax.legend(fontsize=9)

# Right: loss over steps
ax = axes[1]
for lr, color, label in zip(learning_rates, lr_colors, lr_labels):
    hist = gradient_descent_1d(w_init, lr, n_steps)
    ls   = [h[1] for h in hist]
    ls_clipped = np.clip(ls, 0, 70)
    ax.plot(range(n_steps), ls_clipped, color=color, lw=2, label=label, marker='o',
            markersize=3)

ax.set_xlabel('Step')
ax.set_ylabel('Loss (clipped at 70)')
ax.set_title('Loss over training steps', fontweight='bold')
ax.legend(fontsize=9)

plt.suptitle('Learning rate controls everything: too large → chaos, too small → slow',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('lr=1.1 > 1.0 causes divergence: the step overshoots so badly it goes further from the minimum each time.')

---
### Exercise 1: Find the Sweet Spot

Try learning rates `0.01, 0.1, 0.5, 0.9`. Run each for **30 steps** starting from `w_init=0.0`. Plot the loss over time for all four on the same axes.

Which learning rate converges fastest without oscillating?

**Hint:** Call `gradient_descent_1d` and extract the loss values from the returned history.

In [ ]:
def run_experiment(lr):
    """
    Run 30 steps of gradient descent on L(w) = (w-3)^2 starting from w=0.
    Returns a list of loss values, one per step.
    """
    raise NotImplementedError("YOUR TURN: call gradient_descent_1d and return the loss history")

# ── Tests ──────────────────────────────────────────────────────────────────────
lrs_to_test = [0.01, 0.1, 0.5, 0.9]
results = {}
for lr in lrs_to_test:
    losses = run_experiment(lr)
    assert len(losses) == 30, f"Expected 30 loss values for lr={lr}, got {len(losses)}"
    results[lr] = losses

# lr=0.01 is much slower: its final loss should be higher than lr=0.1
assert results[0.01][-1] > results[0.1][-1], \
    f"lr=0.01 should converge slower than lr=0.1 in 30 steps. Got {results[0.01][-1]:.4f} vs {results[0.1][-1]:.4f}"
print('\u2713 passed')

# Plot
colors = [BLUE, GREEN, ORANGE, PURPLE]
fig, ax = plt.subplots(figsize=(10, 4.5))
for lr, color in zip(lrs_to_test, colors):
    ax.plot(range(30), results[lr], color=color, lw=2, label=f'lr={lr}')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Loss over 30 steps for four learning rates', fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

<details>
<summary>💡 Solution</summary>

```python
def run_experiment(lr):
    history = gradient_descent_1d(w_init=0.0, lr=lr, n_steps=30)
    return [loss for (w, loss) in history]
```

**What to observe:** `lr=0.1` converges cleanly in ~20 steps. `lr=0.5` drops fast but may oscillate slightly near the minimum (visible as a tiny uptick). `lr=0.9` oscillates visibly — each step overshoots the minimum and corrects back. `lr=0.01` is slow — it takes all 30 steps and still hasn't bottomed out.

The "sweet spot" depends on the loss surface. For this parabola with curvature 2, the critical learning rate is $\alpha < 1/2 = 0.5$. This is why practitioners say **"tune the learning rate first"** — it matters more than almost anything else.

</details>

---
## Part 3: 2D Loss Surface

Real models have millions of parameters. But the key behaviors — saddle points, ravines, poor conditioning — all show up at 2D. Let's build intuition here before scaling up.

Consider this 2D loss function:

$$L(w_1, w_2) = w_1^2 + 10 \cdot w_2^2$$

This is a **ravine**: the curvature in the $w_2$ direction is 10× stronger than in $w_1$. The gradient is:

$$\nabla L = \left[\frac{\partial L}{\partial w_1}, \frac{\partial L}{\partial w_2}\right] = [2w_1,\ 20w_2]$$

The minimum is at $(0, 0)$. But watch what happens when gradient descent tries to navigate there.

In [ ]:
def L2d(w):
    """2D loss: L(w1, w2) = w1^2 + 10*w2^2"""
    return w[0]**2 + 10 * w[1]**2

def grad_L2d(w):
    """Analytical gradient: [2*w1, 20*w2]"""
    return np.array([2 * w[0], 20 * w[1]])

def gd_2d_demo(w_init, lr, n_steps):
    """Run gradient descent on the 2D ravine loss."""
    w = np.array(w_init, dtype=float)
    trajectory = [w.copy()]
    for _ in range(n_steps):
        grad = grad_L2d(w)
        w = w - lr * grad
        trajectory.append(w.copy())
    return np.array(trajectory)

# Run from (3, 3) with a carefully chosen lr
traj = gd_2d_demo(w_init=[3.0, 3.0], lr=0.09, n_steps=50)
print(f'Starting point: {traj[0]}')
print(f'After 10 steps: {traj[10].round(4)}')
print(f'After 50 steps: {traj[50].round(6)}')
print()
print(f'Loss at start:  {L2d(traj[0]):.4f}')
print(f'Loss at step 10: {L2d(traj[10]):.4f}')
print(f'Loss at step 50: {L2d(traj[50]):.6f}')

In [ ]:
# Contour plot of the loss surface with trajectory
w1 = np.linspace(-3.5, 3.5, 300)
w2 = np.linspace(-3.5, 3.5, 300)
W1, W2 = np.meshgrid(w1, w2)
Z = W1**2 + 10 * W2**2

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Left: contour plot with trajectory
ax = axes[0]
levels = [0.1, 0.5, 1, 2, 4, 8, 15, 25, 40, 60, 90]
cp = ax.contour(W1, W2, Z, levels=levels, cmap='Blues', alpha=0.7)
ax.clabel(cp, inline=True, fontsize=7, fmt='%.0f')

# Draw trajectory
traj_x = traj[:, 0]
traj_y = traj[:, 1]
ax.plot(traj_x, traj_y, color=ORANGE, lw=1.5, alpha=0.8, zorder=4)
ax.scatter(traj_x[::5], traj_y[::5], color=ORANGE, s=40, zorder=5, label='every 5th step')
ax.scatter([traj_x[0]], [traj_y[0]], color=RED, s=120, zorder=6, marker='*', label='start (3, 3)')
ax.scatter([0], [0], color=GREEN, s=120, zorder=6, marker='*', label='minimum (0, 0)')

ax.set_xlabel('$w_1$')
ax.set_ylabel('$w_2$')
ax.set_title('GD trajectory on the ravine\n$L = w_1^2 + 10w_2^2$, lr=0.09',
             fontweight='bold')
ax.legend(fontsize=9)
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)

# Right: loss over steps
ax = axes[1]
losses_over_steps = [L2d(traj[i]) for i in range(len(traj))]
ax.plot(range(len(traj)), losses_over_steps, color=BLUE, lw=2)
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Loss over 50 steps (lr=0.09)', fontweight='bold')
ax.set_yscale('log')

plt.suptitle('Ravine: GD zigzags in the steep direction ($w_2$) while barely moving in $w_1$',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Notice the zigzag: GD oscillates across the ravine in w2 while creeping along w1.')
print('The gradient in w2 (20*w2) dominates and causes overshooting across the steep dimension.')

---
### Exercise 2: The Conditioning Problem

The loss $L(w_1, w_2) = w_1^2 + 10w_2^2$ is **poorly conditioned** — the curvature in $w_2$ is 10× stronger than in $w_1$. A single learning rate must serve both directions, which creates a conflict.

**Your task:** Implement `gd_2d` that runs gradient descent on this loss and returns the full trajectory as a 2D array. Then experiment with `lr=0.05` and `lr=0.15`.

- Which one diverges first?
- What does the trajectory look like?

**Key constraint:** The maximum stable learning rate for this problem is $\alpha < 1/\max(\text{eigenvalue}) = 1/20 = 0.05$. Anything above that and the $w_2$ component diverges.

In [ ]:
def gd_2d(w_init, lr, n_steps):
    """
    Run gradient descent on L(w1, w2) = w1^2 + 10*w2^2.

    Args:
        w_init:  list or array of shape (2,) — starting point
        lr:      learning rate (float)
        n_steps: number of update steps

    Returns:
        trajectory: np.ndarray of shape (n_steps, 2)
                    trajectory[i] is the parameter vector after step i
    """
    raise NotImplementedError("YOUR TURN: implement gradient descent on the 2D ravine")

# ── Tests ──────────────────────────────────────────────────────────────────────
traj_test = gd_2d(w_init=[3.0, 3.0], lr=0.05, n_steps=40)
assert traj_test.shape == (40, 2), \
    f"Expected shape (40, 2), got {traj_test.shape}"
assert traj_test[0, 0] < 3.0, \
    "First step should move w1 toward 0 (negative direction from 3.0)"
print('\u2713 passed')

# Visualize both learning rates
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, lr_val, color, title in [
    (axes[0], 0.04, BLUE,  'lr=0.04 (stable — slow zigzag)'),
    (axes[1], 0.1,  RED,   'lr=0.1  (unstable — diverges in w2)'),
]:
    cp = ax.contour(W1, W2, Z, levels=levels, cmap='Blues', alpha=0.5)
    try:
        traj_v = gd_2d(w_init=[3.0, 3.0], lr=lr_val, n_steps=30)
        tx = np.clip(traj_v[:, 0], -3.5, 3.5)
        ty = np.clip(traj_v[:, 1], -3.5, 3.5)
        ax.plot(tx, ty, color=color, lw=1.5)
        ax.scatter(tx, ty, color=color, s=25, zorder=5)
        ax.scatter([tx[0]], [ty[0]], color=ORANGE, s=120, zorder=6, marker='*', label='start')
        ax.scatter([0], [0], color=GREEN, s=120, zorder=6, marker='*', label='minimum')
    except Exception:
        ax.text(0, 0, 'Diverged', ha='center', va='center', fontsize=14, color=RED)
    ax.set_xlim(-3.5, 3.5)
    ax.set_ylim(-3.5, 3.5)
    ax.set_xlabel('$w_1$')
    ax.set_ylabel('$w_2$')
    ax.set_title(title, fontweight='bold')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

<details>
<summary>💡 Solution</summary>

```python
def gd_2d(w_init, lr, n_steps):
    w = np.array(w_init, dtype=float)
    trajectory = []
    for _ in range(n_steps):
        grad = grad_L2d(w)
        w = w - lr * grad
        trajectory.append(w.copy())
    return np.array(trajectory)
```

**What you should observe:**
- `lr=0.04`: both dimensions shrink steadily. $w_1$ converges slowly; $w_2$ zigzags but shrinks. Stable but inefficient.
- `lr=0.1`: $w_2$ explodes immediately because $0.1 \times 20 = 2 > 1$ — the eigenvalue condition is violated.

**Key insight:** This is exactly why **Adam and RMSprop** exist. They maintain a *per-parameter learning rate* scaled by the gradient history, so steep dimensions ($w_2$) automatically get a smaller effective learning rate. Vanilla SGD forces one rate to serve all dimensions — a fundamental limitation.

</details>

---
## Part 4: Numerical Gradients

In practice you don't compute gradients analytically — PyTorch's autograd does it via the chain rule. But understanding **numerical gradients** is essential for:

1. **Gradient checking**: verifying your autograd implementation is correct
2. **Debugging**: when loss doesn't decrease, is the gradient zero? Exploding? Nan?
3. **Understanding**: the numerical gradient is the definition made executable

The **finite difference** formula approximates the derivative using two nearby function evaluations:

$$\frac{df}{dw} \approx \frac{f(w + \epsilon) - f(w - \epsilon)}{2\epsilon}$$

This is the centered difference formula — accurate to $O(\epsilon^2)$ rather than $O(\epsilon)$ for the one-sided version. With $\epsilon = 10^{-5}$, you typically get 8–10 decimal places of accuracy.

In [ ]:
def numerical_gradient(f, x, eps=1e-5):
    """
    Compute the numerical (finite difference) gradient of scalar function f at point x.

    Uses the centered difference: (f(x + eps) - f(x - eps)) / (2 * eps)

    Args:
        f:   callable, takes a scalar, returns a scalar
        x:   float, the point at which to evaluate the gradient
        eps: float, small perturbation (default 1e-5)

    Returns:
        float: approximate gradient at x
    """
    return (f(x + eps) - f(x - eps)) / (2 * eps)

# Verify on our simple parabola
for w_test in [0.0, 1.5, 3.0, 5.0]:
    num_grad  = numerical_gradient(L, w_test)
    anal_grad = dL(w_test)
    error     = abs(num_grad - anal_grad)
    print(f'w={w_test}: numerical={num_grad:+.8f}  analytical={anal_grad:+.8f}  error={error:.2e}')

print()
print('Errors are ~1e-10: the numerical gradient is extremely accurate.')

---
### Exercise 3: Verify a More Complex Gradient

Verify that `numerical_gradient` matches the analytical gradient for a cubic:

$$L(w) = w^3 - 2w^2 + w$$

The analytical gradient is:

$$\frac{dL}{dw} = 3w^2 - 4w + 1$$

Implement both functions, then assert they agree to within `1e-6` at `w = 2.0`.

**This is called a gradient check** — the same technique PyTorch uses internally in `torch.autograd.gradcheck` to verify that custom backward implementations are correct.

In [ ]:
def L_cubic(w):
    """L(w) = w^3 - 2*w^2 + w"""
    raise NotImplementedError("YOUR TURN: implement the cubic loss function")

def analytical_gradient_cubic(w):
    """dL/dw = 3*w^2 - 4*w + 1"""
    raise NotImplementedError("YOUR TURN: implement the analytical gradient")

# ── Tests ──────────────────────────────────────────────────────────────────────
w_check = 2.0
num_grad  = numerical_gradient(L_cubic, w_check)
anal_grad = analytical_gradient_cubic(w_check)
error     = abs(num_grad - anal_grad)

print(f'At w={w_check}:')
print(f'  Numerical gradient:   {num_grad:.10f}')
print(f'  Analytical gradient:  {anal_grad:.10f}')
print(f'  Error:                {error:.2e}')

assert error < 1e-6, \
    f'Gradients disagree by {error:.2e} — check your implementation'
print('\u2713 passed — gradient check succeeded!')

# Bonus: plot both across a range to see they match everywhere
w_range_cubic = np.linspace(-1, 3, 200)
num_grads  = [numerical_gradient(L_cubic, w) for w in w_range_cubic]
anal_grads = [analytical_gradient_cubic(w) for w in w_range_cubic]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
ax = axes[0]
ax.plot(w_range_cubic, [L_cubic(w) for w in w_range_cubic], color=BLUE, lw=2, label='$L(w) = w^3 - 2w^2 + w$')
ax.set_xlabel('w')
ax.set_ylabel('L(w)')
ax.set_title('Cubic loss function', fontweight='bold')
ax.legend()

ax = axes[1]
ax.plot(w_range_cubic, anal_grads, color=GREEN, lw=2.5, label='Analytical $dL/dw$')
ax.plot(w_range_cubic, num_grads, color=RED, lw=1.5, ls='--', label='Numerical (finite diff)', alpha=0.8)
ax.set_xlabel('w')
ax.set_ylabel('dL/dw')
ax.set_title('Gradients overlap perfectly\n(numerical ≈ analytical)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

<details>
<summary>💡 Solution</summary>

```python
def L_cubic(w):
    return w**3 - 2*w**2 + w

def analytical_gradient_cubic(w):
    return 3*w**2 - 4*w + 1
```

At `w=2.0`: the analytical gradient is $3(4) - 4(2) + 1 = 12 - 8 + 1 = 5$. The numerical estimate should be `5.0000000xxxxxxx` — the error lives in the 10th decimal place, far below our `1e-6` threshold.

**In practice:** `torch.autograd.gradcheck` does exactly this — it perturbs each input dimension by $\epsilon$, measures the output change, and compares to the backward-pass gradient. If your custom `autograd.Function` passes gradient check, it's almost certainly correct.

</details>

---
## Part 5: Mini-Batch Gradient Descent

**Full-batch GD** computes the gradient over *all* training examples each step — accurate but expensive ($O(n)$ per step).

**Stochastic GD (SGD)** uses *one* random example per step — cheap but very noisy.

**Mini-batch GD** is the practical compromise: compute the gradient on a small random batch (typically 32–512 examples). This is both:
- **Stable**: averaging over a batch reduces gradient variance vs. single-sample SGD
- **Fast**: batches map naturally to GPU parallelism — a batch of 256 takes the same wall-clock time as a batch of 1 on modern hardware

We'll apply this to linear regression: learn $w$ and $b$ in $\hat{y} = wx + b$.

The MSE loss and its gradients:

$$L = \frac{1}{n}\sum_i(\hat{y}_i - y_i)^2 = \frac{1}{n}\|Xw + b - y\|^2$$

$$\frac{\partial L}{\partial w} = \frac{2}{n} X^T(Xw + b - y), \quad \frac{\partial L}{\partial b} = \frac{2}{n}\sum_i(Xw_i + b - y_i)$$

In [ ]:
# Generate synthetic linear regression data: y = 3x + 2 + noise
np.random.seed(42)
n = 1000
X = np.random.randn(n, 1)
y = 3 * X.squeeze() + 2 + np.random.randn(n) * 0.5   # true w=3, true b=2

print(f'Dataset: {n} samples')
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'True relationship: y = 3x + 2 + noise')
print(f'y range: [{y.min():.2f}, {y.max():.2f}]')

# Quick scatter
fig, ax = plt.subplots(figsize=(8, 4))
idx = np.random.choice(n, 200, replace=False)
ax.scatter(X[idx], y[idx], color=BLUE, alpha=0.4, s=15, label='data')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Linear regression dataset\n(we will learn y = wx + b from scratch)', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def mse_loss_and_grad(X_batch, y_batch, w, b):
    """
    Compute MSE loss and gradients for linear regression.

    Args:
        X_batch: shape (batch_size, 1)
        y_batch: shape (batch_size,)
        w:       float — weight
        b:       float — bias

    Returns:
        loss:   scalar MSE loss
        dw:     gradient w.r.t. w
        db:     gradient w.r.t. b
    """
    n_batch   = len(y_batch)
    y_pred    = X_batch.squeeze() * w + b       # predictions: shape (batch_size,)
    residuals = y_pred - y_batch                  # shape (batch_size,)

    loss = np.mean(residuals ** 2)
    dw   = (2 / n_batch) * np.dot(X_batch.squeeze(), residuals)
    db   = (2 / n_batch) * np.sum(residuals)

    return loss, dw, db


def mini_batch_gd(X, y, w_init, b_init, lr, batch_size, n_epochs):
    """
    Train a linear model y = wx + b using mini-batch gradient descent.

    Returns:
        w_history:    list of w values per epoch
        b_history:    list of b values per epoch
        loss_history: list of mean batch loss per epoch
    """
    w = w_init
    b = b_init
    n = len(y)
    w_history    = []
    b_history    = []
    loss_history = []

    for epoch in range(n_epochs):
        # Shuffle data each epoch
        perm = np.random.permutation(n)
        X_shuffled = X[perm]
        y_shuffled = y[perm]

        epoch_losses = []
        for start in range(0, n, batch_size):
            X_batch = X_shuffled[start:start + batch_size]
            y_batch = y_shuffled[start:start + batch_size]

            loss, dw, db = mse_loss_and_grad(X_batch, y_batch, w, b)
            w = w - lr * dw
            b = b - lr * db
            epoch_losses.append(loss)

        w_history.append(w)
        b_history.append(b)
        loss_history.append(np.mean(epoch_losses))

    return w_history, b_history, loss_history


# Train with batch_size=32, lr=0.05, 60 epochs
np.random.seed(0)
w_hist, b_hist, loss_hist = mini_batch_gd(
    X, y,
    w_init=0.0, b_init=0.0,
    lr=0.05, batch_size=32, n_epochs=60
)

print(f'After 60 epochs:')
print(f'  Learned w = {w_hist[-1]:.4f}  (true: 3.0)')
print(f'  Learned b = {b_hist[-1]:.4f}  (true: 2.0)')
print(f'  Final loss: {loss_hist[-1]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
ax.plot(loss_hist, color=BLUE, lw=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Mini-batch GD training loss\n(batch_size=32, lr=0.05)', fontweight='bold')

ax = axes[1]
idx = np.random.choice(n, 200, replace=False)
ax.scatter(X[idx], y[idx], color=BLUE, alpha=0.3, s=15, label='data')
x_line = np.linspace(X.min(), X.max(), 100)
ax.plot(x_line, w_hist[-1] * x_line + b_hist[-1],
        color=RED, lw=2.5, label=f'learned: y={w_hist[-1]:.2f}x+{b_hist[-1]:.2f}')
ax.plot(x_line, 3 * x_line + 2,
        color=GREEN, lw=2, ls='--', label='true: y=3x+2')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Learned line vs true line', fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
### Exercise 4: Batch Size Effect

Train the same linear model with batch sizes **1, 32, and 256**. Run each for 40 epochs with `lr=0.05`. Plot the loss-per-epoch curve for all three.

**What to observe:**
- `batch_size=1` (pure SGD): very noisy — each step uses only one example
- `batch_size=32`: moderate noise, fast convergence
- `batch_size=256`: smooth curve but fewer gradient steps per epoch

All three should reach final MSE < 1.0.

In [ ]:
def run_batch_experiment(batch_size, n_epochs=40, lr=0.05):
    """
    Train the linear model with the given batch_size.
    Returns (w_history, b_history, loss_history).
    """
    raise NotImplementedError("YOUR TURN: call mini_batch_gd with the given batch_size")

# ── Tests ──────────────────────────────────────────────────────────────────────
np.random.seed(7)
batch_results = {}
for bs in [1, 32, 256]:
    w_h, b_h, l_h = run_batch_experiment(bs)
    assert len(l_h) == 40, f"Expected 40 epoch losses for batch_size={bs}"
    assert l_h[-1] < 1.0, f"batch_size={bs} did not converge (final loss={l_h[-1]:.4f})"
    batch_results[bs] = (w_h, b_h, l_h)
print('\u2713 passed')

# Verify batch_size=1 is noisier than batch_size=256 (higher std of loss history)
std_sgd  = np.std(batch_results[1][2])
std_full = np.std(batch_results[256][2])
assert std_sgd > std_full, \
    f'batch_size=1 should be noisier than batch_size=256. std(bs=1)={std_sgd:.4f}, std(bs=256)={std_full:.4f}'
print('\u2713 batch_size=1 is noisier than batch_size=256 as expected')

# Plot
batch_colors = {1: RED, 32: BLUE, 256: GREEN}
fig, ax = plt.subplots(figsize=(10, 4.5))
for bs, color in batch_colors.items():
    l_h = batch_results[bs][2]
    ax.plot(l_h, color=color, lw=2, alpha=0.85, label=f'batch_size={bs}')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Effect of batch size on training stability\n(all converge, but noise differs)', fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

print('\nFinal losses:')
for bs in [1, 32, 256]:
    print(f'  batch_size={bs:4d}: {batch_results[bs][2][-1]:.4f}')

<details>
<summary>💡 Solution</summary>

```python
def run_batch_experiment(batch_size, n_epochs=40, lr=0.05):
    np.random.seed(7)
    return mini_batch_gd(
        X, y,
        w_init=0.0, b_init=0.0,
        lr=lr, batch_size=batch_size, n_epochs=n_epochs
    )
```

**What to observe:**
- `batch_size=1` (SGD): jagged loss curve — each gradient estimate is from a single noisy example. Surprisingly, it often converges faster in wall-clock time because it takes many steps per epoch.
- `batch_size=256`: smooth, monotonically decreasing loss — gradient estimates are accurate. But takes fewer steps per epoch (1000/256 ≈ 4 batches vs. 1000 batches for bs=1).
- `batch_size=32`: the sweet spot in practice — enough smoothing to be stable, enough steps to converge quickly.

**Why batch_size=32 is the default in most code:** It fits in GPU L1 cache efficiently and provides good gradient estimates. Larger batches require learning rate scaling (linear scaling rule) to maintain the same effective update magnitude.

</details>

---
## Part 6: Momentum

Plain SGD has two problems:
1. **Oscillation in ravines**: it zigzags back and forth across steep dimensions
2. **Slow progress along flat directions**: gradients in shallow dimensions are small

**Momentum** fixes both. Instead of only following the current gradient, you also carry forward a running average of past gradients — a "velocity" that accumulates in consistent directions and dampens oscillations.

$$v_{t+1} = \mu \cdot v_t - \alpha \cdot \nabla L(w_t)$$
$$w_{t+1} = w_t + v_{t+1}$$

where $\mu$ is the momentum coefficient (typically 0.9 — keep 90% of previous velocity).

Think of it as a ball rolling down a hill: it accumulates speed in the downhill direction and resists direction changes.

In [ ]:
def sgd_with_momentum(grad_fn, w_init, lr, momentum, n_steps):
    """
    SGD with momentum on a 1D loss.

    Args:
        grad_fn:  callable, takes w, returns gradient at w
        w_init:   float, starting parameter value
        lr:       float, learning rate
        momentum: float, momentum coefficient (e.g. 0.9)
        n_steps:  int, number of update steps

    Returns:
        history: list of (w, L(w)) tuples
    """
    w        = w_init
    velocity = 0.0
    history  = []
    for _ in range(n_steps):
        history.append((w, L(w)))
        grad     = grad_fn(w)
        velocity = momentum * velocity - lr * grad
        w        = w + velocity
    return history

# Compare plain GD vs momentum on the 1D parabola starting from a bad init
w_start  = 0.0
n_steps  = 40

hist_gd  = gradient_descent_1d(w_start, lr=0.1, n_steps=n_steps)
hist_mom = sgd_with_momentum(dL, w_start, lr=0.1, momentum=0.9, n_steps=n_steps)

loss_gd  = [h[1] for h in hist_gd]
loss_mom = [h[1] for h in hist_mom]

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
ax.plot(range(n_steps), loss_gd,  color=BLUE,  lw=2, label='Plain GD (lr=0.1)')
ax.plot(range(n_steps), loss_mom, color=ORANGE, lw=2, label='Momentum (lr=0.1, μ=0.9)')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Momentum vs plain GD: loss over steps', fontweight='bold')
ax.legend(fontsize=10)

ax = axes[1]
ws_gd  = [h[0] for h in hist_gd]
ws_mom = [h[0] for h in hist_mom]
ax.plot(ws_range := np.linspace(-0.5, 6.5, 200), L(ws_range), color='#888', lw=2, alpha=0.4)
ax.plot(ws_gd[:15],  L(np.array(ws_gd[:15])),  color=BLUE,   lw=2, marker='o', ms=4, label='Plain GD')
ax.plot(ws_mom[:15], L(np.array(ws_mom[:15])), color=ORANGE, lw=2, marker='s', ms=4, label='Momentum')
ax.axvline(3, color='gray', lw=1, ls='--')
ax.set_xlabel('w')
ax.set_ylabel('L(w)')
ax.set_title('Trajectory on loss surface (first 15 steps)', fontweight='bold')
ax.legend(fontsize=10)

plt.suptitle('Momentum accumulates velocity in the consistent direction → faster convergence',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'Plain GD  — loss after {n_steps} steps: {loss_gd[-1]:.6f}')
print(f'Momentum  — loss after {n_steps} steps: {loss_mom[-1]:.6f}')

---
### Exercise 5: Adam in 20 Lines

**Adam** = Momentum + RMSprop. It maintains:
- **First moment** $m$: running mean of gradients (like momentum)
- **Second moment** $v$: running mean of squared gradients (like RMSprop)

Then it scales each update by its own gradient history — steep dimensions automatically get smaller steps.

$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t \quad \text{(first moment)}$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2 \quad \text{(second moment)}$$
$$\hat{m}_t = \frac{m_t}{1 - \beta_1^t} \quad \text{(bias correction)}$$
$$\hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$
$$\Delta w = -\frac{\alpha \cdot \hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

Defaults: $\alpha=0.001$, $\beta_1=0.9$, $\beta_2=0.999$, $\epsilon=10^{-8}$.

**Your task:** Implement `adam_step` that computes one Adam update, then run it on the 2D ravine and compare to plain GD.

In [ ]:
def adam_step(grad, m, v, t, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
    """
    Compute one Adam update step.

    Args:
        grad:  current gradient (scalar or array)
        m:     first moment estimate (same shape as grad)
        v:     second moment estimate (same shape as grad)
        t:     current timestep (int, 1-indexed)
        lr, beta1, beta2, eps: Adam hyperparameters

    Returns:
        update: the parameter update Δw (add this to w)
        new_m:  updated first moment
        new_v:  updated second moment
    """
    raise NotImplementedError("YOUR TURN: implement the Adam update step")

# ── Tests ──────────────────────────────────────────────────────────────────────
# Test with a scalar gradient
g_test = 2.0
m_test = 0.0
v_test = 0.0

update, new_m, new_v = adam_step(g_test, m_test, v_test, t=1)

assert isinstance(update, (float, np.floating)), \
    f'update should be a float, got {type(update)}'
assert abs(new_m - (1 - 0.9) * g_test) < 1e-9, \
    f'First moment incorrect: expected {(1-0.9)*g_test:.4f}, got {new_m:.4f}'
assert abs(new_v - (1 - 0.999) * g_test**2) < 1e-9, \
    f'Second moment incorrect: expected {(1-0.999)*g_test**2:.6f}, got {new_v:.6f}'
assert update < 0, 'Update should be negative (we subtract from w for positive grad)'
print('\u2713 passed scalar test')

# Test with array gradient (2D case)
g_arr = np.array([2.0, 20.0])
m_arr = np.zeros(2)
v_arr = np.zeros(2)
update_arr, new_m_arr, new_v_arr = adam_step(g_arr, m_arr, v_arr, t=1)
assert update_arr.shape == (2,), f'Expected shape (2,), got {update_arr.shape}'
print('\u2713 passed array test')

# Key check: Adam gives a SMALLER update in the steep dimension
# (because v_hat in w2 direction is larger)
print(f'\nGradients:    w1={g_arr[0]:.1f}, w2={g_arr[1]:.1f}  (w2 is 10x steeper)')
print(f'Adam updates: w1={update_arr[0]:.5f}, w2={update_arr[1]:.5f}')
print('Adam reduces the w2 update relative to w1 — per-parameter scaling at work.')

In [ ]:
# Run Adam on the 2D ravine and compare to plain GD
def run_adam_2d(w_init, lr=0.5, n_steps=100):
    """Run Adam on L(w1, w2) = w1^2 + 10*w2^2."""
    w   = np.array(w_init, dtype=float)
    m   = np.zeros(2)
    v   = np.zeros(2)
    traj = [w.copy()]
    for t in range(1, n_steps + 1):
        grad          = grad_L2d(w)
        update, m, v  = adam_step(grad, m, v, t=t, lr=lr)
        w             = w + update
        traj.append(w.copy())
    return np.array(traj)

traj_adam = run_adam_2d(w_init=[3.0, 3.0], lr=0.5, n_steps=100)
traj_gd50 = gd_2d_demo(w_init=[3.0, 3.0], lr=0.04, n_steps=100)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

# Contour + trajectories
ax = axes[0]
cp = ax.contour(W1, W2, Z, levels=levels, cmap='Blues', alpha=0.6)
ax.clabel(cp, inline=True, fontsize=7, fmt='%.0f')

for traj_v, color, label in [
    (traj_gd50, BLUE,   'Plain GD (lr=0.04)'),
    (traj_adam, ORANGE, 'Adam (lr=0.5)'),
]:
    tx = np.clip(traj_v[:, 0], -3.5, 3.5)
    ty = np.clip(traj_v[:, 1], -3.5, 3.5)
    ax.plot(tx, ty, color=color, lw=1.5, alpha=0.8)
    ax.scatter(tx[::10], ty[::10], color=color, s=35, zorder=5)
    ax.scatter([tx[0]], [ty[0]], color=color, s=120, zorder=6, marker='*')

ax.scatter([0], [0], color=GREEN, s=150, zorder=7, marker='*', label='minimum')
ax.set_xlim(-3.5, 3.5)
ax.set_ylim(-3.5, 3.5)
ax.set_xlabel('$w_1$')
ax.set_ylabel('$w_2$')
ax.set_title('Adam navigates the ravine cleanly\nPlain GD zigzags', fontweight='bold')

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color=BLUE,   lw=2, label='Plain GD (lr=0.04)'),
    Line2D([0], [0], color=ORANGE, lw=2, label='Adam (lr=0.5)'),
    Line2D([0], [0], color=GREEN,  lw=0, marker='*', ms=10, label='minimum'),
]
ax.legend(handles=legend_elements, fontsize=9)

# Loss over steps
ax = axes[1]
losses_gd   = [L2d(traj_gd50[i])  for i in range(101)]
losses_adam = [L2d(traj_adam[i]) for i in range(101)]
ax.plot(losses_gd,   color=BLUE,   lw=2, label='Plain GD (lr=0.04)')
ax.plot(losses_adam, color=ORANGE, lw=2, label='Adam (lr=0.5)')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_yscale('log')
ax.set_title('Loss over 100 steps (log scale)', fontweight='bold')
ax.legend(fontsize=10)

plt.suptitle('Adam: per-parameter learning rates solve the ravine problem',
             fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'After 100 steps:')
print(f'  Plain GD loss: {losses_gd[-1]:.6f}')
print(f'  Adam loss:     {losses_adam[-1]:.8f}')
print(f'\nAdam converges orders of magnitude faster on poorly-conditioned surfaces.')

<details>
<summary>💡 Solution</summary>

```python
def adam_step(grad, m, v, t, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
    # Update biased moments
    new_m = beta1 * m + (1 - beta1) * grad
    new_v = beta2 * v + (1 - beta2) * grad ** 2

    # Bias correction
    m_hat = new_m / (1 - beta1 ** t)
    v_hat = new_v / (1 - beta2 ** t)

    # Parameter update
    update = -lr * m_hat / (np.sqrt(v_hat) + eps)

    return update, new_m, new_v
```

**Why bias correction?** At $t=1$, $m_1 = (1 - 0.9) \cdot g_1 = 0.1 g_1$ — heavily biased toward zero because the moment was initialized at 0. Dividing by $(1 - \beta_1^t) = 0.1$ cancels this bias. By $t=100$, $(1 - 0.9^{100}) \approx 1$, so bias correction has almost no effect.

**Why Adam handles ravines:** In the steep $w_2$ direction, $v$ accumulates large squared gradients $(20w_2)^2$. The $\sqrt{\hat{v}} + \epsilon$ denominator grows large, shrinking the effective step size. In the shallow $w_1$ direction, $\sqrt{\hat{v}}$ stays small, allowing larger effective steps. This is automatic per-parameter learning rate adaptation.

</details>

---
## Connecting Forward

Everything you just built by hand — gradient computation, update rules, mini-batching, momentum, Adam — is what PyTorch autograd does automatically.

| What you built | PyTorch equivalent |
|---|---|
| `dL(w)` (analytical derivative) | `loss.backward()` via chain rule |
| `numerical_gradient(f, x)` | `torch.autograd.gradcheck(fn, x)` |
| `gradient_descent_1d` | `torch.optim.SGD(model.parameters(), lr=0.1)` |
| `sgd_with_momentum` | `torch.optim.SGD(..., momentum=0.9)` |
| `adam_step` × n_steps | `torch.optim.Adam(model.parameters(), lr=0.001)` |
| mini-batch loop | `torch.utils.data.DataLoader(dataset, batch_size=32)` |

When you call `loss.backward()` in PyTorch, it is computing the gradient we coded manually in Exercise 3 — just via automatic differentiation through a computation graph rather than by hand. When you use `torch.optim.Adam`, it is the 20-line function from Exercise 5, applied simultaneously to every parameter in the model.

**The mathematics is identical.** PyTorch just makes it scale to millions of parameters without you having to derive every gradient by hand.

### What's next

In the Deep Learning course, you'll apply these exact algorithms to neural networks:
- A single neuron: $f(x) = \sigma(wx + b)$ — the same gradient descent, one layer deep
- A 2-layer network: gradient descent + chain rule (backpropagation)
- A full MLP: what happens when you stack 10 layers and 10 million parameters

The 2D ravine you navigated in Exercise 2? Real neural network loss surfaces have billions of dimensions. Adam's per-parameter scaling is why training is possible at all.

---
*Notebook 6 of the Deep Learning series — [ML Edge](https://mle-edge.dev) self-directed curriculum*